# 3DGS Colab 示例（gsplat 版）

官方 2023 引擎在当前 Colab（Python 3.13 / torch 2.11 / CUDA 13）上编不过 `diff_gaussian_rasterization`，所以这版改用还在维护的 gsplat（nerfstudio-project/gsplat）官方 `examples/simple_trainer.py`，默认策略、30000 步。

- 训练产物：`results/gsplat_truck/ply/point_cloud_29999.ply`（30000 步对应的最后一步），拖进 SuperSplat 就能旋转查看
- 评估：训练在 7000 和 30000 步做 held-out 评估，输出 PSNR / SSIM / LPIPS，见训练日志和 `results/gsplat_truck/stats/`
- 运行时必须是 **GPU**，这里用 T4；`--data_factor 1` 是全分辨率，效果最好，T4 上如果显存不够再报出来改 2，不要悄悄降画质
- 旧版官方引擎 notebook 仍在 `notebooks/3dgs_colab.ipynb`，本版不覆盖它

In [ ]:
import torch, subprocess
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout)

In [ ]:
# 克隆本仓库和 gsplat：已存在时也更新到 main 最新版
import os, subprocess
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f'command failed: {cmd}')
if not os.path.exists('/content/3dgs-colab/scripts/download_example.py'):
    sh('git clone https://github.com/miaopasi677/3dgs-colab.git /content/3dgs-colab')
else:
    sh('git fetch origin && git checkout main && git reset --hard origin/main', cwd='/content/3dgs-colab')
if not os.path.exists('/content/gsplat/examples/simple_trainer.py'):
    sh('git clone --recursive https://github.com/nerfstudio-project/gsplat.git /content/gsplat')
else:
    sh('git fetch origin && git checkout main && git pull --ff-only && git submodule update --init --recursive', cwd='/content/gsplat')
print('repo ok:', os.path.exists('/content/3dgs-colab/scripts/download_example.py'))
print('gsplat trainer ok:', os.path.exists('/content/gsplat/examples/simple_trainer.py'))
print('gsplat HEAD:', subprocess.run('git rev-parse --short HEAD', shell=True, cwd='/content/gsplat', capture_output=True, text=True).stdout.strip())
import glob
print('glm files:', glob.glob('/content/gsplat/gsplat/cuda/csrc/third_party/glm/**/type_ptr.hpp', recursive=True)[:3])


In [ ]:
# 安装 gsplat：官方要求 torch>=2.7，当前 Colab 的 torch 2.11 + CUDA 13 正好在其支持范围（README 明确写了 CUDA 13 支持）
# 注意：不要装 gsplat 官方 examples/requirements.txt 原样，那里面钉了 torch==2.9.1，会把 Colab 的 torch 降级；这里只装训练真正要的依赖，不动 torch
%cd /content/gsplat
# gsplat 导入时会即时编译 C++ 扩展，必须先装 ninja，否则会报 Ninja is required（上一轮就是缺它，不是 gsplat 本身不兼容）
!pip install ninja
!which ninja
!pip install -e .
!pip install pycolmap viser torchmetrics tyro "imageio[ffmpeg]" opencv-python-headless scikit-learn scipy tensorboard pyyaml matplotlib Pillow piexif tensorly splines
!pip install git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787
!pip install git+https://github.com/rahul-goel/fused-ssim@a7c48d6dd7ac6dc39a7958c7c4452e0b10418f38
!pip install git+https://github.com/harry7557558/fused-bilagrid@49f0ef06c9f81810fb9b5dd9027cf1844950cc16 || echo "fused-bilagrid optional, not needed for default training"
!pip install git+https://github.com/nv-tlabs/ppisp@v1.2.1 || echo "ppisp install failed - only needed for ppisp post-processing, not for default training"
# 导入校验：gsplat 核心 + 官方 trainer 的导入（trainer 顶部会引 viewer 相关包，缺了会在这里明说，不要等训练才报错）
!python -c "import gsplat; print('gsplat import OK', gsplat.__version__ if hasattr(gsplat, '__version__') else '')"
import sys
sys.path.insert(0, '/content/gsplat/examples')
import simple_trainer
print('simple_trainer import OK')


In [ ]:
# 下载官方示例：Tanks and Temples truck，自带 COLMAP，不用先跑 COLMAP
%cd /content/3dgs-colab
!python scripts/download_example.py
import os
print('truck dir:', os.path.exists('data/tandt/truck/images'), os.path.exists('data/tandt/truck/sparse/0'))


In [ ]:
# 训练：gsplat 官方 simple_trainer，default 策略，30000 步，全分辨率
# 默认 eval/save/ply 步数都是 [7000, 30000]，所以 7000 步会先出一次评估和 ply，30000 步出最终结果
# --disable_viewer 和 --disable_video 是为了在 Colab 无头环境下不打开网页 viewer、不生成视频
%cd /content/gsplat/examples
!python simple_trainer.py default --data_dir /content/3dgs-colab/data/tandt/truck --data_factor 1 --result_dir /content/3dgs-colab/results/gsplat_truck --max_steps 30000 --save_ply --disable_viewer --disable_video
print('training cell finished - check the next cell for ply and metrics')


In [ ]:
# 找产物：ply、ckpt、评估指标、渲染图
import pathlib, glob
base = pathlib.Path('/content/3dgs-colab/results/gsplat_truck')
for p in sorted(base.rglob('*')):
    if p.is_file():
        print(f"{p.relative_to(base)}  {p.stat().st_size/1024/1024:.1f} MB")
print('---')
ply = sorted(base.rglob('*.ply'))
print('ply files:', [str(p) for p in ply])
if ply:
    print('main ply (largest):', max(ply, key=lambda p: p.stat().st_size))


In [ ]:
# 展示几张渲染图（如果评估/轨迹渲染生成了图片）
from IPython.display import display, Image
import pathlib, glob
base = pathlib.Path('/content/3dgs-colab/results/gsplat_truck')
imgs = sorted(base.rglob('*.png'))[:6]
print('render images found:', len(imgs))
for p in imgs:
    print(p)
    display(Image(str(p)))
